# Parte 3 — Operação (semanas 17 a 20)

**Banco Aurora · Prevenção a Fraudes** · entrega: **segunda-feira, 16/11, 23h59**

Leia antes o `demanda_parte3.md`, só o e-mail principal. Os anexos têm hora para serem lidos.

É a mesma operação da parte 2: quatro semanas, uma por vez, e uma decisão a cada semana. Desta vez, as contestações da última semana ficam com a Helena, que vai usá-las para conferir o modelo que você deixar em produção.

**Como este notebook funciona**
- Uma seção por semana, na ordem do roteiro.
- No fim ficam as seções **I**nvestigação, **D**ecisão, **R**etreino, **A**ção e **P**lano de monitoramento. Use-as quando e se precisar. A seção P é obrigatória.
- O sistema é operado pelas **telas**. Se quiser automatizar pelo notebook, as rotas da API estão em `http://127.0.0.1:8000/docs`; nesse caso, o código é seu.
- Duas células vêm prontas: a de ambiente e a de entrega.

**O que será avaliado nesta parte** (a nota descreve o **estado** do que você entregou)
1. O modelo em produção respeita o contrato de features, no treino e na inferência.
2. O que está em produção é rastreável: MLflow, contrato e `CHANGELOG.md` contam a mesma história.
3. O monitoramento existia e foi usado **toda semana**, antes do diagnóstico.
4. O diagnóstico do que aconteceu está correto.
5. A ação é proporcional e bem executada. Se houve retreino: dados certos, validação temporal, comparação justa com o modelo atual e threshold pelo custo.
6. A comunicação com o negócio é clara, incluindo o balanço da operação e o plano de monitoramento.

A Helena também vai pontuar o modelo final com as contestações da semana 20, que só ela tem.

## 0. Ambiente

Rode a célula abaixo. Ela confere as conexões e lista as versões do `antifraude` no seu MLflow.

In [1]:
# CÉLULA PRONTA — só rode. Tudo precisa aparecer como "ok".
import json
import urllib.request

import mlflow
import psycopg2
from mlflow.tracking import MlflowClient

import config

for nome, uri in [("Postgres / antifraude", config.URI_BANCO_APP), ("Postgres / mlflow", config.URI_BANCO_MLFLOW)]:
    try:
        psycopg2.connect(uri).close()
        print(f"{nome:24} ok")
    except Exception as e:
        print(f"{nome:24} FALHOU -> {e}")

for nome, url in [("MLflow", f"{config.MLFLOW_TRACKING_URI}/health"), ("Sistema", f"{config.SISTEMA_URL}/saude")]:
    try:
        resp = urllib.request.urlopen(url, timeout=5).read().decode()
        print(f"{nome:24} ok -> {resp[:160]}")
    except Exception as e:
        print(f"{nome:24} FALHOU -> {e}")

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
cli = MlflowClient()
print("\nversões de", config.NOME_MODELO, "no MLflow:")
for v in sorted(cli.search_model_versions(f"name='{config.NOME_MODELO}'"), key=lambda v: int(v.version)):
    alias = ", ".join("@" + a for a in v.aliases)
    print(f"  v{v.version:<3} {alias:12} threshold={v.tags.get(config.TAG_THRESHOLD, '—'):6} {v.tags.get(config.TAG_MOTIVO, '')}")

Postgres / antifraude    ok
Postgres / mlflow        ok
MLflow                   ok -> OK


Sistema                  ok -> {"sistema":"ok","contrato":"1.1","modelo":{"ok":true,"versao":"9","threshold":0.05,"run_id":"d81b7759854042a0a939e3f2832af1d6","descricao":"Especialista boostin

versões de antifraude no MLflow:
  v1                threshold=0.29   v1.0.0 - primeiro modelo em produção
  v2                threshold=0.29   v1.0.0 - primeiro modelo em produção
  v3                threshold=0.29   v1.0.0 - primeiro modelo em produção
  v4                threshold=0.29   v1.0.0 - primeiro modelo em produção
  v5                threshold=0.09   Menor custo na comparação ampliada das semanas 11 e 12
  v6                threshold=0.09   Menor custo na comparação ampliada das semanas 11 e 12
  v7                threshold=0.09   Menor custo na comparação ampliada das semanas 11 e 12
  v8                threshold=0.05   estado oficial pós-parte 2 (v1.0.1)
  v9                threshold=0.05   Mitigar deterioração de detecção no Pix


## 1. Antes do primeiro lote

### 1.1 Ponto de partida
Revise como a sua parte 2 terminou:
- O `@champion` trata corretamente o que mudou nas semanas 13 a 16?
- O contrato em `contrato/` está na versão certa?
- A referência do monitoramento representa o "normal" atual?
- As semanas 13 a 16 estão no sistema, com os rótulos de 13 a 15? Os rótulos da 16 chegam agora e precisam do lote 16 no sistema.

**Você escolhe com o que vai operar:**
- **Com o que você tem.** Vale se a sua parte 2 ficou certa.
- **Com o estado oficial** (`estado_oficial_parte2/`). Vale se a sua parte 2 ficou errada ou incompleta. Usar o estado oficial **não tira ponto** nesta parte. O passo a passo está no `estado_oficial_parte2/LEIAME.md`.

Registre a escolha e o motivo na decisão da semana 17. Se trocar o champion ou o contrato, isso entra no `CHANGELOG.md`.

### 1.2 O relatório da semana
Reaproveite o relatório da parte 2 e confira:
- ele carrega a **referência vigente** do champion, a do MLflow;
- ele compara o que **o modelo vê**;
- ele mede a performance por segmento.

Pergunte-se também: **se a performance cair sem nenhuma feature mudar, o seu relatório avisa?**

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

EVIDENCIAS = Path("monitoramento/parte3")
if not EVIDENCIAS.is_dir():
    EVIDENCIAS = Path("evidencias_operacao")

def ler(arquivo):
    return pd.read_csv(EVIDENCIAS / arquivo)

def evidencia_semana(semana):
    pasta = f"semana_{semana}"
    continuas = ler(f"{pasta}/continuas.csv")
    discretas = ler(f"{pasta}/discretas.csv")
    saida = ler(f"{pasta}/saida.csv")
    display(pd.DataFrame([{
        "semana": semana, "maior_KS_D": continuas["D"].max(),
        "KS_score": continuas.loc[continuas["variavel"].eq("score"), "D"].iat[0],
        "maior_mudanca_discreta_pp": discretas["maior_mudanca_pp"].max(),
        "alertas_features_e_score": int(continuas["alerta"].sum() + discretas["alerta"].sum()),
    }]))
    display(saida[["semana","versao_modelo","taxa_bloqueio","valor_bloqueado","alerta"]])

def desempenho_disponivel(semana):
    desempenho = ler(f"semana_{semana}/performance.csv")
    colunas = ["semana","versao_modelo","valor_segmento","fraudes","recall","precision","custo","motivos_alerta"]
    display(desempenho.loc[desempenho["semana"].isna() | desempenho["semana"].ge(semana - 3), colunas])


In [3]:
from app import contrato

print("Contrato em produção:", contrato.carregar()["versao"])
print("As tabelas semanais abaixo são os relatórios gravados antes de cada decisão.")
print("Automação de ingestão e monitoramento: python operar_parte3.py <semana>.")


Contrato em produção: 1.1
As tabelas semanais abaixo são os relatórios gravados antes de cada decisão.
Automação de ingestão e monitoramento: python operar_parte3.py <semana>.


**Ponto de partida.** Adotei o estado oficial pós-Parte 2, registrado como `antifraude v8` (corte 0,05, contrato v1.1). Minha Parte 2 converteu os lotes 15 e 16 na base. A v7 publicada ainda recebia dados no horário de Brasília. A v8 converte hora e dia em UTC no próprio pipeline, e a referência de sua run está no mesmo fuso que o modelo vê.

A API confirma os lotes 13 a 16 e os rótulos da 13 à 15 antes do primeiro envio. Preservamos a pontuação histórica desses lotes. Repontuar os lotes 15 e 16 diretamente com a v8 aplicaria a conversão duas vezes. Nos relatórios, cada semana rotulada é medida com os scores e o corte usados naquela semana.


## Semana 17

**Chegam:** `dados/parte3/transacoes_semana_17.csv` e `dados/parte3/rotulos_semana_16.csv`.

1. Envie o lote na tela **Lotes** e, **depois**, os rótulos na tela **Rótulos**. Os dados já pontuados saem na tela **Lotes**, no link *baixar dados*.
2. Rode o seu relatório da semana:
   - drift do lote 17 contra a referência vigente (features e score);
   - saída do modelo (taxa de bloqueio e valor bloqueado), comparada com a referência e com as semanas anteriores;
   - **performance da semana 16**, que acabou de ganhar rótulo: recall, precision e custo, globais **e por segmento**. Compare com a referência e com as semanas anteriores.
3. Escreva abaixo, em 2 a 4 linhas: **o que você viu e o que vai fazer**.
4. Registre a decisão da semana 17 na tela **Decisões**.

Se decidir agir agora, faça a investigação e a ação (seções I, D, R e A) **antes** de seguir para a próxima semana.

In [4]:
evidencia_semana(17)


,semana,maior_KS_D,KS_score,maior_mudanca_discreta_pp,alertas_features_e_score
0,17,0.06396,0.011553,1.003355,0


,semana,versao_modelo,taxa_bloqueio,valor_bloqueado,alerta
0,NaN,8,0.026964,499547.33,False
1,13.0,7,0.026520,300237.57,False
2,14.0,7,0.039643,391360.35,True
3,15.0,7,0.025222,270299.69,False
4,16.0,7,0.027356,273064.10,False
5,17.0,8,0.024302,231043.40,False


In [5]:
desempenho_disponivel(17)


,semana,versao_modelo,valor_segmento,fraudes,recall,precision,custo,motivos_alerta
0,NaN,8,todos,600,0.836667,0.471362,28448.23,NaN
1,NaN,8,cartao_online,201,0.885572,0.557994,9351.57,NaN
2,NaN,8,cartao_presencial,113,0.716814,0.338912,8307.79,NaN
3,NaN,8,pix,286,0.849650,0.479290,10788.87,NaN
8,14.0,7,todos,303,0.841584,0.314039,15065.62,"queda_precision, queda_pr_auc"
9,14.0,7,cartao_online,107,0.841121,0.276923,6175.85,queda_precision
10,14.0,7,cartao_presencial,68,0.823529,0.327485,3077.49,NaN
11,14.0,7,pix,128,0.851562,0.344937,5812.28,queda_precision
12,15.0,7,todos,247,0.813765,0.397233,9759.94,queda_precision
13,15.0,7,cartao_online,77,0.805195,0.317949,3752.14,NaN


**Semana 17.** A v8 registrou 489 retenções (2,43%) no lote 17, sem alerta de distribuição ou saída. Os rótulos da 16 mostraram recall global de 83,17% e custo de R$ 13.912,46. A precisão de cartão online, de 42,33%, acionou alerta quando comparada à referência vigente. Mantive a v8 e acompanhei o segmento na semana seguinte.


## Semana 18

**Chegam:** `dados/parte3/transacoes_semana_18.csv` e `dados/parte3/rotulos_semana_17.csv`.

1. Envie o lote na tela **Lotes** e, **depois**, os rótulos na tela **Rótulos**. Os dados já pontuados saem na tela **Lotes**, no link *baixar dados*.
2. Rode o seu relatório da semana:
   - drift do lote 18 contra a referência vigente (features e score);
   - saída do modelo (taxa de bloqueio e valor bloqueado), comparada com a referência e com as semanas anteriores;
   - **performance da semana 17**, que acabou de ganhar rótulo: recall, precision e custo, globais **e por segmento**. Compare com a referência e com as semanas anteriores.
3. Escreva abaixo, em 2 a 4 linhas: **o que você viu e o que vai fazer**.
4. Registre a decisão da semana 18 na tela **Decisões**.

Se decidir agir agora, faça a investigação e a ação (seções I, D, R e A) **antes** de seguir para a próxima semana.

> **Antes da semana 19:** leia o **Anexo 1** da demanda.

In [6]:
evidencia_semana(18)


,semana,maior_KS_D,KS_score,maior_mudanca_discreta_pp,alertas_features_e_score
0,18,0.059589,0.011375,0.573869,0


,semana,versao_modelo,taxa_bloqueio,valor_bloqueado,alerta
0,NaN,8,0.026964,499547.33,False
1,13.0,7,0.026520,300237.57,False
2,14.0,7,0.039643,391360.35,True
3,15.0,7,0.025222,270299.69,False
4,16.0,7,0.027356,273064.10,False
5,17.0,8,0.024302,231043.40,False
6,18.0,8,0.026966,231729.74,False


In [7]:
desempenho_disponivel(18)


,semana,versao_modelo,valor_segmento,fraudes,recall,precision,custo,motivos_alerta
0,NaN,8,todos,600,0.836667,0.471362,28448.23,NaN
1,NaN,8,cartao_online,201,0.885572,0.557994,9351.57,NaN
2,NaN,8,cartao_presencial,113,0.716814,0.338912,8307.79,NaN
3,NaN,8,pix,286,0.849650,0.479290,10788.87,NaN
12,15.0,7,todos,247,0.813765,0.397233,9759.94,queda_precision
13,15.0,7,cartao_online,77,0.805195,0.317949,3752.14,NaN
14,15.0,7,cartao_presencial,57,0.842105,0.358209,2264.98,NaN
15,15.0,7,pix,113,0.805310,0.514124,3742.82,NaN
16,16.0,7,todos,315,0.831746,0.479853,13912.46,NaN
17,16.0,7,cartao_online,112,0.812500,0.423256,6486.02,queda_precision


**Semana 18.** A taxa de bloqueio do lote 18 foi de 2,70%, dentro dos limites de saída. Com as contestações da 17, o recall global foi de 82,14%, a precisão de cartão online voltou a 56,62% e o custo foi R$ 16.786,97. Mantive a v8.


## Semana 19

**Chegam:** `dados/parte3/transacoes_semana_19.csv` e `dados/parte3/rotulos_semana_18.csv`.

1. Envie o lote na tela **Lotes** e, **depois**, os rótulos na tela **Rótulos**. Os dados já pontuados saem na tela **Lotes**, no link *baixar dados*.
2. Rode o seu relatório da semana:
   - drift do lote 19 contra a referência vigente (features e score);
   - saída do modelo (taxa de bloqueio e valor bloqueado), comparada com a referência e com as semanas anteriores;
   - **performance da semana 18**, que acabou de ganhar rótulo: recall, precision e custo, globais **e por segmento**. Compare com a referência e com as semanas anteriores.
3. Escreva abaixo, em 2 a 4 linhas: **o que você viu e o que vai fazer**.
4. Registre a decisão da semana 19 na tela **Decisões**.

Se decidir agir agora, faça a investigação e a ação (seções I, D, R e A) **antes** de seguir para a próxima semana.

> **Antes da semana 20:** leia o **Anexo 2** da demanda.

In [8]:
evidencia_semana(19)


,semana,maior_KS_D,KS_score,maior_mudanca_discreta_pp,alertas_features_e_score
0,19,0.064052,0.01131,1.06406,0


,semana,versao_modelo,taxa_bloqueio,valor_bloqueado,alerta
0,NaN,8,0.026964,499547.33,False
1,13.0,7,0.026520,300237.57,False
2,14.0,7,0.039643,391360.35,True
3,15.0,7,0.025222,270299.69,False
4,16.0,7,0.027356,273064.10,False
5,17.0,8,0.024302,231043.40,False
6,18.0,8,0.026966,231729.74,False
7,19.0,8,0.027163,246682.20,False


In [9]:
desempenho_disponivel(19)


,semana,versao_modelo,valor_segmento,fraudes,recall,precision,custo,motivos_alerta
0,NaN,8,todos,600,0.836667,0.471362,28448.23,NaN
1,NaN,8,cartao_online,201,0.885572,0.557994,9351.57,NaN
2,NaN,8,cartao_presencial,113,0.716814,0.338912,8307.79,NaN
3,NaN,8,pix,286,0.849650,0.479290,10788.87,NaN
16,16.0,7,todos,315,0.831746,0.479853,13912.46,NaN
17,16.0,7,cartao_online,112,0.812500,0.423256,6486.02,queda_precision
18,16.0,7,cartao_presencial,59,0.796610,0.415929,3526.14,NaN
19,16.0,7,pix,144,0.861111,0.568807,3900.30,NaN
20,17.0,8,todos,280,0.821429,0.470348,16786.97,NaN
21,17.0,8,cartao_online,94,0.819149,0.566176,4852.25,NaN


**Semana 19.** Depois do Anexo 1, os rótulos da 18 revelaram custo de R$ 44.205,70 e detecção de 43,46% no canal Pix. No perfil de Pix em aparelho conhecido com pelo menos duas transações na hora anterior, foram detectadas 14 das 168 fraudes. Mantive a v8 durante a preparação de um candidato e reservei a semana 19 para confirmá-lo em dados de outra semana.


## Semana 20

**Chegam:** `dados/parte3/transacoes_semana_20.csv` e `dados/parte3/rotulos_semana_19.csv`.

1. Envie o lote na tela **Lotes** e, **depois**, os rótulos na tela **Rótulos**. Os dados já pontuados saem na tela **Lotes**, no link *baixar dados*.
2. Rode o seu relatório da semana:
   - drift do lote 20 contra a referência vigente (features e score);
   - saída do modelo (taxa de bloqueio e valor bloqueado), comparada com a referência e com as semanas anteriores;
   - **performance da semana 19**, que acabou de ganhar rótulo: recall, precision e custo, globais **e por segmento**. Compare com a referência e com as semanas anteriores.
3. Escreva abaixo, em 2 a 4 linhas: **o que você viu e o que vai fazer**.
4. Registre a decisão da semana 20 na tela **Decisões**.

Se decidir agir agora, faça a investigação e a ação (seções I, D, R e A) **antes** de seguir para a próxima semana.

> As contestações da semana 20 não chegam: quem vai conferir é a Helena. O que você sabe da semana 20 vem das features e do score.

In [10]:
evidencia_semana(20)


,semana,maior_KS_D,KS_score,maior_mudanca_discreta_pp,alertas_features_e_score
0,20,0.070517,0.013529,0.987201,0


,semana,versao_modelo,taxa_bloqueio,valor_bloqueado,alerta
0,NaN,8,0.026964,499547.33,False
1,13.0,7,0.026520,300237.57,False
2,14.0,7,0.039643,391360.35,True
3,15.0,7,0.025222,270299.69,False
4,16.0,7,0.027356,273064.10,False
5,17.0,8,0.024302,231043.40,False
6,18.0,8,0.026966,231729.74,False
7,19.0,8,0.027163,246682.20,False
8,20.0,8,0.024712,205451.68,False


In [11]:
desempenho_disponivel(20)


,semana,versao_modelo,valor_segmento,fraudes,recall,precision,custo,motivos_alerta
0,NaN,8,todos,600,0.836667,0.471362,28448.23,NaN
1,NaN,8,cartao_online,201,0.885572,0.557994,9351.57,NaN
2,NaN,8,cartao_presencial,113,0.716814,0.338912,8307.79,NaN
3,NaN,8,pix,286,0.849650,0.479290,10788.87,NaN
20,17.0,8,todos,280,0.821429,0.470348,16786.97,NaN
21,17.0,8,cartao_online,94,0.819149,0.566176,4852.25,NaN
22,17.0,8,cartao_presencial,55,0.727273,0.310078,4370.11,NaN
23,17.0,8,pix,131,0.862595,0.504464,7564.61,NaN
24,18.0,8,todos,482,0.597510,0.520796,44205.70,"queda_recall, queda_pr_auc, aumento_custo_por_mil"
25,18.0,8,cartao_online,113,0.920354,0.615385,1937.97,NaN


**Semana 20.** Os rótulos da 19 confirmaram a perda no mesmo grupo. O custo observado foi R$ 43.639,74. A v9 validada reduziria esse custo para R$ 16.269,41 em comparação nas mesmas transações. Promovi a v9 e repontuei o lote 20, ainda sem rótulos. As retenções passaram de 491 para 815, com as mudanças restritas ao grupo.


## I. Investigação

Para cada sinal (do monitoramento, do painel ou dos e-mails), responda com evidência:

1. **O que mudou?** Nas features, na saída do modelo ou só na performance?
2. **Desde quando?** Em que semana começou? A mudança é abrupta ou gradual?
3. **Onde?** Em todas as transações ou em um grupo? Como você identifica esse grupo **só com as 13 features do contrato**?
4. **Que tipo de problema é?** Mudou o **dado**, mudou o **mundo** (a distribuição das features) ou mudou a **relação** entre as features e a fraude? Quais hipóteses você testou, e como descartou as outras?
5. **Quanto custa?** Mostre custo, bloqueio e recall por semana, global e no grupo afetado.

> Lembre: um monitor de features verde não quer dizer que o modelo está bem. E um monitor vermelho não quer dizer que ele está mal.

In [12]:
perfil = ler("semana_20/perfil_pix.csv")
selecionado = perfil["conjunto"].eq("lote") & perfil["semana"].between(16, 19) & perfil["segmento"].eq("perfil")
grupos = perfil.loc[selecionado].assign(
    prevalencia_fraude=lambda p: p["fraudes"] / p["transacoes"],
)
display(grupos[["semana","valor_segmento","transacoes","fraudes","prevalencia_fraude","recall","precision","custo"]])


,semana,valor_segmento,transacoes,fraudes,prevalencia_fraude,recall,precision,custo
17,16,outros_canais,10995,171,0.015553,0.807018,0.420732,10012.16
18,16,outros_pix,8459,129,0.015250,0.844961,0.576720,3690.30
19,16,pix_dispositivo_conhecido_2mais_1h,505,15,0.029703,1.000000,0.517241,210.00
21,17,outros_canais,11093,149,0.013432,0.785235,0.441509,9222.36
22,17,outros_pix,8597,124,0.014424,0.879032,0.526570,6468.01
23,17,pix_dispositivo_conhecido_2mais_1h,432,7,0.016204,0.571429,0.235294,1096.60
25,18,outros_canais,11260,176,0.015631,0.880682,0.516667,7513.72
26,18,outros_pix,8665,138,0.015926,0.862319,0.526549,7147.86
27,18,pix_dispositivo_conhecido_2mais_1h,582,168,0.288660,0.083333,0.518519,29544.12
29,19,outros_canais,10432,178,0.017063,0.820225,0.505190,8175.19


In [13]:
faixas = ler("semana_19/faixas_perfil_pix.csv")
display(faixas[faixas["perfil"].eq("pix_dispositivo_conhecido_2mais_1h")])


,semana,perfil,faixa_valor,transacoes,fraudes,fraudes_bloqueadas,taxa_fraude,taxa_bloqueio,fraude_perdida
10,17,pix_dispositivo_conhecido_2mais_1h,"[0.0, 100.0)",227,2,1,0.008811,0.026432,36.48
11,17,pix_dispositivo_conhecido_2mais_1h,"[100.0, 250.0)",108,0,0,0.000000,0.027778,0.00
12,17,pix_dispositivo_conhecido_2mais_1h,"[250.0, 500.0)",75,3,1,0.040000,0.040000,865.12
13,17,pix_dispositivo_conhecido_2mais_1h,"[500.0, 1000.0)",18,1,1,0.055556,0.111111,0.00
14,17,pix_dispositivo_conhecido_2mais_1h,"[1000.0, inf)",4,1,1,0.250000,0.750000,0.00
25,18,pix_dispositivo_conhecido_2mais_1h,"[0.0, 100.0)",265,67,2,0.252830,0.037736,4025.87
26,18,pix_dispositivo_conhecido_2mais_1h,"[100.0, 250.0)",190,53,2,0.278947,0.021053,8453.96
27,18,pix_dispositivo_conhecido_2mais_1h,"[250.0, 500.0)",84,31,4,0.369048,0.071429,9222.11
28,18,pix_dispositivo_conhecido_2mais_1h,"[500.0, 1000.0)",34,11,2,0.323529,0.088235,5178.63
29,18,pix_dispositivo_conhecido_2mais_1h,"[1000.0, inf)",9,6,4,0.666667,0.444444,2468.55


In [14]:
comparacao = ler("validacao/validacao_semana_19.csv")
display(comparacao.query("modelo == 'champion'")[[
    "segmento","fraude_perdida","fraudes_detectadas","legitimas_bloqueadas","taxa_bloqueio","custo"
]])


,segmento,fraude_perdida,fraudes_detectadas,legitimas_bloqueadas,taxa_bloqueio,custo
0,global,39784.74,268,257,0.027163,43639.74
1,grupo_pix,31693.32,14,10,0.043321,31843.32
2,outros_pix,2061.23,108,104,0.025414,3621.23
3,outros_canais,6030.19,146,143,0.027703,8175.19
4,restante,8091.42,254,247,0.026686,11796.42


In [15]:
display(ler("impacto_fuso.csv"))


,semana,cenario,custo,bloqueadas,legitimas_bloqueadas,recall
0,14,utc_sem_conversao,15065.62,812,557,0.841584
1,14,horario_brasilia,10525.93,553,293,0.858086
2,15,utc_sem_conversao,14503.03,750,557,0.781377
3,15,horario_brasilia,9759.94,506,305,0.813765
4,16,utc_sem_conversao,18714.78,803,540,0.834921
5,16,horario_brasilia,13912.46,546,284,0.831746


**Diagnóstico.** A quebra ficou evidente nos rótulos da 18, com mudança entre as semanas 17 e 18. A proporção de fraudes no grupo subiu de 7/432 (1,62%) para 168/582 (28,87%) e 176/554 (31,77%) na 19. Nesse grupo, o recall da v8 caiu de 57,14% para 8,33% e 7,95%. Entre os outros Pix, ficou em 87,90%, 86,23% e 87,10%. Dos R$ 44.205,70 de custo na 18, R$ 29.544,12 vieram do grupo.

O grupo subiu de 2,15% para 2,84% do volume total da 17 para a 18. Os testes de distribuição e de score, feitos antes das decisões, permaneceram dentro dos limites, assim como as retenções globais da v8. A mudança central foi na chance de fraude para um conjunto de sinais já conhecido. A hipótese de erro de fuso é enfraquecida pela conversão dentro do pipeline, pelo controle de hora após a transformação e pelo desempenho dos outros Pix. O perfil é compatível com o golpe descrito. Confirmar acesso remoto exigiria informações adicionais da investigação.


## D. Decisão

| Alternativa | Avaliação |
|---|---|
| Manter a versão vigente | Cabia na 17 e na 18, com os dados disponíveis. Na 19, manter temporariamente permitiu validar na semana seguinte. Na 20, a nova queda justificou mudar. |
| Mudar apenas o corte | Testei corte global e corte específico para o grupo. O corte global aumentou o custo na 19. O corte por grupo melhorou, mas custou R$ 889,26 a mais e gerou 94 retenções legítimas adicionais em relação ao especialista. |
| Corrigir dados ou pré-processamento | O fuso já é tratado pela v8 e foi verificado com o monitoramento e a API. |
| Atualizar apenas a referência | Uma nova referência seria útil para comparar a v9. A atualização isolada não mudaria as transações aprovadas. |
| Mudar o contrato | Os sinais necessários já estão entre as 13 features. Mantive a v1.1. |
| Retreinar | Treinei um especialista para o grupo afetado. Na semana de validação, ele reduziu o custo global com decisões idênticas no restante do lote. |


**Escolha.** Promover a v9 no fim da semana 20 e repontuar o lote 20. A v9 mantém o corte 0,05 na API. Dentro do grupo, o score do especialista é remapeado para aplicar seu próprio corte, escolhido pelo custo.


## R. Retreino (se você decidiu retreinar)

Retreinar é **criar um candidato e provar que ele é melhor** que o champion. Responda e mostre:

1. **Quais dados.** Com quais semanas você treina, e por quê? Está usando todo o histórico ou só o recente? Os dados de semanas diferentes têm o **mesmo formato e o mesmo significado** que o modelo espera na entrada? Confira as versões do contrato ao longo do tempo.
2. **Validação temporal.** Treine com as semanas mais antigas e valide nas **mais recentes com rótulo**, que o candidato não viu. Nada de divisão aleatória.
3. **Comparação justa.** Champion e candidato nas **mesmas semanas de validação**: custo, recall e taxa de bloqueio, globais **e no grupo afetado**. O candidato não pode piorar o resto para melhorar o grupo.
4. **Threshold pelo custo**, escolhido na validação, como na parte 1.
5. **Modelo final.** Você vai usar o candidato validado ou retreinar com tudo, incluindo a validação, mantendo o threshold? Justifique.
6. **Plano de volta.** Qual versão volta se o novo se comportar mal, e o que dispara essa volta?
**Execução.** A amostra de ajuste reúne 2.666 transações do grupo nas semanas 13 a 17 e nos cinco primeiros dias da 18, com 192 fraudes. O histórico fornece operações legítimas e fraudes anteriores. A parte inicial da 18 traz o padrão recente. O arquivo da 13 foi harmonizado para UTC e os arquivos das semanas 14 em diante foram lidos nas versões originais da Vértice. O pipeline converte todas para Brasília antes de pontuar.

O fim de semana da 18, com 176 transações e 49 fraudes do grupo, serviu para calibrar pelo custo e comparar floresta e boosting. Depois de escolher o boosting e congelar os cortes, usei a semana 19 inteira para validar. Conservei o modelo validado para manter a configuração efetivamente medida. Na 19, a taxa de retenção subiu de 2,72% para 4,40%, o recall global subiu de 56,07% para 88,28% e o custo do restante ficou em R$ 11.796,42 com ambas as versões.

**Volta à v8.** Voltar se a API divergir da versão registrada ou se, com rótulos completos, a v9 custar mais do que a v8 nas mesmas transações. Mover o alias `champion` para v8, conferir as previsões na API, repontuar apenas lotes em aberto que precisam da correção e registrar a ação.


In [16]:
amostras = ler("validacao/amostras.csv")
display(amostras[amostras["grupo"]])


,uso,semana,grupo,count,sum
1,calibracao,18,True,176,49
3,treino,13,True,438,15
5,treino,14,True,469,21
7,treino,15,True,416,15
9,treino,16,True,505,15
11,treino,17,True,432,7
13,treino,18,True,406,119


In [17]:
calibracao = ler("validacao/calibracao_semana_18.csv")
display(calibracao.query("segmento in ['global','grupo_pix']")[
    ["modelo","segmento","corte","custo","recall","precisao","taxa_bloqueio"]
])


,modelo,segmento,corte,custo,recall,precisao,taxa_bloqueio
0,champion,global,0.050000,13612.08,0.635220,0.564246,0.030489
1,champion,grupo_pix,0.050000,9173.80,0.142857,0.700000,0.056818
5,corte_global,global,0.016727,11640.79,0.685535,0.367003,0.050588
6,corte_global,grupo_pix,0.016727,7855.85,0.183673,0.600000,0.085227
10,corte_por_grupo,global,0.000907,5893.28,0.899371,0.453968,0.053654
11,corte_por_grupo,grupo_pix,0.000907,1455.00,1.000000,0.335616,0.829545
15,floresta_pix,global,0.046262,5180.60,0.886792,0.570850,0.042071
16,floresta_pix,grupo_pix,0.046262,742.32,0.959184,0.602564,0.443182
20,boosting_pix,global,0.004266,5079.46,0.893082,0.552529,0.043774
21,boosting_pix,grupo_pix,0.004266,641.18,0.979592,0.545455,0.500000


In [18]:
display(comparacao.query("segmento in ['global','grupo_pix','restante']")[
    ["modelo","segmento","corte","custo","recall","precisao","taxa_bloqueio"]
])


,modelo,segmento,corte,custo,recall,precisao,taxa_bloqueio
0,champion,global,0.050000,43639.74,0.560669,0.510476,0.027163
1,champion,grupo_pix,0.050000,31843.32,0.079545,0.583333,0.043321
4,champion,restante,0.050000,11796.42,0.841060,0.506986,0.026686
5,corte_global,global,0.016727,47488.14,0.596234,0.291411,0.050600
6,corte_global,grupo_pix,0.016727,31422.55,0.090909,0.390244,0.074007
9,corte_global,restante,0.016727,16065.59,0.890728,0.287086,0.049909
10,corte_por_grupo,global,0.000907,17158.67,0.884937,0.447146,0.048945
11,corte_por_grupo,grupo_pix,0.000907,5362.25,0.960227,0.379775,0.803249
14,corte_por_grupo,restante,0.000907,11796.42,0.841060,0.506986,0.026686
15,boosting_pix,global,0.004266,16269.41,0.882845,0.495887,0.044029


In [19]:
custos = comparacao.query("segmento == 'global'").set_index("modelo")["custo"]
print(f"Redução simulada de custo na semana 19: R$ {custos['champion'] - custos['boosting_pix']:,.2f}")


Redução simulada de custo na semana 19: R$ 27,370.33


In [20]:
# Para reproduzir a seleção e a validação com o simulador e os dados originais,
# execute na raiz do projeto: python treinar_parte3.py --versao-base 8


## A. Ação e rastreabilidade

- **Mudou o modelo ou o pipeline:**
  - registre uma nova versão com as tags `threshold` e `motivo` e uma descrição;
  - inclua `code_paths=["features.py"]`, se o pipeline usa código seu;
  - mova o `@champion`;
  - refaça o teste `POST /predict` × modelo do MLflow;
  - decida, e justifique, quais semanas **repontuar** (botão *Repontuar* na tela **Lotes**).
- **Mudou o contrato:** suba a `versao` e registre a mudança no `changelog` do próprio YAML.
- **Mudou a referência:** grave-a como artefato da run do champion, em `referencia/`, e diga qual é a vigente.
- **Sempre:**
  - uma entrada no `CHANGELOG.md` por versão nova;
  - o **antes e depois** nas semanas afetadas.
**Execução.** O alias passou para a v9, na run `d81b7759854042a0a939e3f2832af1d6`. O teste da API incluiu seis transações do grupo e seis do restante. A referência `referencia_semana_19.csv` contém scores da v9 e hora e dia após a conversão para Brasília. Mantivemos os registros das semanas 13 a 19 para preservar os resultados observados. Apenas o lote 20 foi repontuado, com as duas pontuações disponíveis no histórico.


In [21]:
display(ler("semana_20/apos_promocao/antes_depois.csv"))


,versao,segmento,transacoes,bloqueadas,taxa_bloqueio,valor_bloqueado
0,8,global,19869,491,0.024712,205451.68
1,8,grupo_pix,574,30,0.052265,15129.32
2,8,restante,19295,461,0.023892,190322.36
3,9,global,19869,815,0.041019,262804.20
4,9,grupo_pix,574,354,0.616725,72481.84
5,9,restante,19295,461,0.023892,190322.36


In [22]:
teste = ler("semana_20/apos_promocao/teste_predict.csv")
display(teste[["id_transacao","grupo_pix","versao_modelo","score","score_mlflow","bloquear"]])


,id_transacao,grupo_pix,versao_modelo,score,score_mlflow,bloquear
0,T20000000,False,9,0.028986,0.028986,False
1,T20000001,False,9,0.002249,0.002249,False
2,T20000002,False,9,0.001677,0.001677,False
3,T20000003,False,9,0.001825,0.001825,False
4,T20000004,False,9,0.001646,0.001646,False
5,T20000005,False,9,0.017376,0.017376,False
6,T20000017,True,9,0.021032,0.021032,False
7,T20000071,True,9,0.006130,0.006130,False
8,T20000214,True,9,0.059726,0.059726,True
9,T20000225,True,9,0.934267,0.934267,True


In [23]:
import requests

resposta = requests.get(f"{config.SISTEMA_URL}/api/modelo", timeout=30)
resposta.raise_for_status()
estado = resposta.json()
assert str(estado["modelo"]["versao"]) == "9"
assert estado["contrato"] == "1.1"
print(estado["modelo"])


{'ok': True, 'versao': '9', 'threshold': 0.05, 'run_id': 'd81b7759854042a0a939e3f2832af1d6', 'descricao': 'Especialista boosting_pix para Pix em aparelho conhecido com 2+ transações na hora anterior. Corte calibrado por custo: 0.0042658561. Fora do grupo, mantém scores e decisões da v8. Validação temporal na semana 19; contrato v1.1.'}


## P. Plano de monitoramento (obrigatório)

O que você deixaria rodando toda semana, daqui para frente? Monte uma tabela com o **sinal**, a **frequência**, o **limite** e a **ação** que o limite dispara.

Leve em conta tudo o que você viu da semana 13 à 20:
- **Sinais sem rótulo:** features, score e taxa de bloqueio.
- **Sinais com rótulo**, que chegam uma semana depois: performance global e por segmento.
- **Sinais do negócio:** e-mails, atendimento, ouvidoria.

Diga também quais segmentos você acompanha separadamente, e por quê.

| Sinal | Frequência | Limite | Ação |
|---|---|---|---|
| Campos, horário e dia da semana | A cada lote | Qualquer violação do contrato ou hora/dia incoerentes com o timestamp em UTC | Verificar a origem com Plataforma e corrigir antes de liberar a pontuação. |
| Features após conversão e score | Semanal, no recebimento | KS D > 0,10 em contínuas ou score, ou mudança > 5 pontos percentuais em uma categoria | Localizar as variáveis e os grupos responsáveis, checar a qualidade e avaliar o impacto da mudança. |
| Taxa e valor retidos | Semanal, no recebimento | Variação global acima de 1 ponto percentual na taxa ou acima de 3 pontos percentuais na parcela do valor | Verificar o modelo e a capacidade de atendimento. Separar o volume do grupo Pix do restante. |
| Recall, precisão e custo por mil | Semanal, na chegada dos rótulos | Recall ou precisão global > 5 pontos percentuais abaixo da referência, custo por mil > 50% acima ou queda > 10 pontos percentuais em um canal com pelo menos 100 fraudes | Investigar por canal, quantificar fraude perdida e retenções legítimas, avisar Prevenção a Fraudes. |
| Desempenho do grupo Pix investigado | Semanal, com rótulos | Recall < 80% com pelo menos 30 fraudes. Se a amostra for menor, agregar as duas últimas semanas já rotuladas e revisar as perdas de maior valor | Revisar cobertura e custo do especialista. Se a v9 custar mais que a v8 nas mesmas transações, acionar o plano de volta. |
| Ouvidoria, Atendimento e alertas setoriais | Triagem diária, resumo semanal | Reclamações de Pix não reconhecido ≥ 2 vezes a média das quatro semanas anteriores, ou relatos convergentes do mesmo padrão | Alertar Prevenção a Fraudes no dia e cruzar os relatos com os scores dos lotes recentes para orientar revisão. |

Acompanhar Pix, cartão online e cartão presencial separadamente. Dentro de Pix, separar aparelho conhecido com duas ou mais transações na hora anterior. A queda se concentrou nesse grupo e pode passar despercebida nas médias do canal. Monitorar também os demais Pix e os cartões para detectar mudanças fora do grupo.


## Comunicação

A resposta à Helena está em `resposta_parte3.md`. O balanço abaixo compara os custos observados com o cenário de aprovar todas as transações. A economia simulada da validação da v9 foi mantida fora do total observado.


In [24]:
display(ler("balanco_13_19.csv")[["semana","fraude_evitada","legitimas_bloqueadas","custo_total","economia_liquida_vs_aprovar_tudo"]])

,semana,fraude_evitada,legitimas_bloqueadas,custo_total,economia_liquida_vs_aprovar_tudo
0,13,195957.33,274.0,10986.30,191847.33
1,14,161158.94,557.0,15065.62,152803.94
2,15,136283.72,305.0,9759.94,131708.72
3,16,163715.93,284.0,13912.46,159455.93
4,17,153926.81,259.0,16786.97,150041.81
5,18,169879.53,265.0,44205.70,165904.53
6,19,173890.07,257.0,43639.74,170035.07


## Entrega

1. Confira a checklist:
   - [ ] as semanas 17 a 20 estão no sistema, e os rótulos de 16 a 19 também;
   - [ ] há uma decisão registrada por semana, de 17 a 20;
   - [ ] o `@champion` tem as tags `threshold` e `motivo`, uma descrição e a `referencia/` na run;
   - [ ] o `CHANGELOG.md` cita a versão que está em produção e a versão do contrato;
   - [ ] o `resposta_parte3.md` está escrito;
   - [ ] a seção P está preenchida;
   - [ ] as células de texto deste notebook estão respondidas.
2. Preencha o seu nome, **salve o notebook** e rode a célula abaixo. Ela gera a pasta `entrega_parte3/` e o `entrega_parte3_<nome>.zip`.
3. Entregue o `.zip` na atividade. Se a célula listar **avisos**, resolva antes de entregar.

> O `modelo_exportado/` que vai na entrega é o que a Helena vai pontuar com a semana 20. Confira que ele é o seu champion final.

In [ ]:
# CÉLULA PRONTA — gera a pasta entrega_parte3/ e o entrega_parte3_<nome>.zip.
# 1) Preencha o seu nome.  2) SALVE o notebook (Ctrl+S).  3) Rode.
NOME_ALUNO = "Levy Guilherme Lemos Rios"

import datetime
import json
import re
import shutil
from pathlib import Path

import io

import mlflow
import pandas as pd
import requests
import yaml
from mlflow.tracking import MlflowClient

import config


def _api(rota):
    r = requests.get(config.SISTEMA_URL.rstrip("/") + rota, timeout=120)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text)) if rota.endswith(".csv") else r.json()


SEMANAS = [17, 18, 19, 20]
ROTULOS = [16, 17, 18, 19]

assert NOME_ALUNO.strip(), "Preencha NOME_ALUNO antes de rodar."
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
cli = MlflowClient()
mv = cli.get_model_version_by_alias(config.NOME_MODELO, config.ALIAS_PRODUCAO)
versoes = sorted(cli.search_model_versions(f"name='{config.NOME_MODELO}'"), key=lambda v: int(v.version))
art_ref = [a.path for a in cli.list_artifacts(mv.run_id, "referencia")]
sist = _api("/api/modelo")
lotes = pd.DataFrame(_api("/api/lotes"))
decisoes = pd.DataFrame(_api("/api/decisoes"))
contrato = yaml.safe_load(Path(config.CONTRATO).read_text(encoding="utf-8"))
changelog = Path("CHANGELOG.md").read_text(encoding="utf-8")
resposta = Path("resposta_parte3.md").read_text(encoding="utf-8")

avisos = []
recebidas = set(lotes["semana"]) if len(lotes) else set()
for s in SEMANAS:
    if s not in recebidas:
        avisos.append(f"a semana {s} não foi enviada ao sistema")
for s in ROTULOS:
    linha = lotes[lotes["semana"] == s] if len(lotes) else lotes
    if linha.empty or int(linha["rotulos"].iloc[0]) != int(linha["qtd_linhas"].iloc[0]):
        avisos.append(f"os rótulos da semana {s} não foram enviados")
com_decisao = set(decisoes["semana"]) if len(decisoes) else set()
for s in SEMANAS:
    if s not in com_decisao or int(decisoes["semana"].eq(s).sum()) != 1:
        avisos.append(f"a semana {s} deve ter exatamente uma decisão registrada")
for chave in (config.TAG_THRESHOLD, config.TAG_MOTIVO):
    if chave not in mv.tags:
        avisos.append(f"a versão champion (v{mv.version}) não tem a tag '{chave}'")
if not mv.description:
    avisos.append(f"a versão champion (v{mv.version}) não tem descrição")
if not art_ref:
    avisos.append("a run do champion não tem artefatos em 'referencia/'")
if str(sist["modelo"].get("versao")) != str(mv.version):
    avisos.append("o sistema não está usando a mesma versão que o MLflow aponta como champion")
if not re.search(rf"{config.NOME_MODELO}\s+v{mv.version}\b", changelog):
    avisos.append(f"o CHANGELOG.md não cita a versão em produção ('{config.NOME_MODELO} v{mv.version}')")
if str(contrato["versao"]) != "1.0" and str(contrato["versao"]) not in changelog:
    avisos.append(f"o contrato está na versão {contrato['versao']}, mas o CHANGELOG.md não cita essa versão")
texto_resposta = re.sub(r"<!--.*?-->", "", resposta, flags=re.S)
if len(texto_resposta.split()) < 80:
    avisos.append("resposta_parte3.md parece vazia ou curta demais")

evidencias = {
    "aluno": NOME_ALUNO.strip(),
    "gerado_em": datetime.datetime.now().isoformat(timespec="seconds"),
    "champion": {"versao": mv.version, "run_id": mv.run_id, "descricao": mv.description, "tags": dict(mv.tags)},
    "versoes": [{"versao": v.version, "aliases": list(v.aliases), "run_id": v.run_id, "criada_em": v.creation_timestamp,
                 "descricao": v.description, "tags": dict(v.tags)} for v in versoes],
    "referencia_champion": art_ref,
    "contrato": {"versao": contrato["versao"], "regras": contrato.get("regras") or [],
                 "changelog": contrato.get("changelog") or []},
    "sistema": sist,
    "avisos": avisos,
}

pasta = Path("entrega_parte3")
if pasta.resolve().parent != Path.cwd().resolve() or pasta.is_symlink():
    raise ValueError("Pasta de entrega fora da raiz do projeto.")
if pasta.exists():
    shutil.rmtree(pasta)
pasta.mkdir()
(pasta / "evidencias_mlflow.json").write_text(json.dumps(evidencias, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
_api("/api/painel.csv").to_csv(pasta / "painel.csv", index=False)
decisoes.to_csv(pasta / "decisoes.csv", index=False)
lotes.to_csv(pasta / "lotes.csv", index=False)
mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{config.NOME_MODELO}@{config.ALIAS_PRODUCAO}",
                                    dst_path=str(pasta / "modelo_exportado"))
if art_ref:
    mlflow.artifacts.download_artifacts(run_id=mv.run_id, artifact_path="referencia", dst_path=str(pasta))
for arq in ["parte3.ipynb", "features.py", "monitoramento.py", "CHANGELOG.md", "resposta_parte3.md",
            "contrato/contrato_features.yaml", "operar_parte3.py", "treinar_parte3.py"]:
    shutil.copy(arq, pasta / Path(arq).name)

shutil.copytree(Path("monitoramento/parte3"), pasta / "evidencias_operacao",
                ignore=shutil.ignore_patterns("referencia_semana_19.csv"))

nome_zip = "entrega_parte3_" + "_".join(NOME_ALUNO.lower().split())
shutil.make_archive(nome_zip, "zip", pasta)
print(f"champion: {config.NOME_MODELO} v{mv.version} | threshold {mv.tags.get(config.TAG_THRESHOLD)} | contrato v{contrato['versao']}")
print(f"decisões registradas: {len(decisoes)} | trocas de modelo no sistema: {len(sist['trocas'])}")
print("AVISOS:" if avisos else "Nenhum aviso.", *avisos, sep="\n  - ")
print(f"\nEntregue o arquivo {nome_zip}.zip")